# Etapa I — Integración y Obtención de Datos Maestros
**Proyecto:** Inteligencia Deportiva — Cruz Azul F.C.  
**Materia:** Calidad y Preprocesamiento de Datos  
**Framework:** DAMA-DMBOK  

Esta notebook cubre la **primera etapa del pipeline**:
1. Estandarización de las 4 fuentes heterogéneas a formato tabular
2. Definición del **modelo canónico** de jugador y partido
3. **Record linkage** para reconciliar nombres de jugadores entre fuentes
4. **Deduplicación** de registros (doble nacionalidad, renombres de equipo)
5. **Fusión** en datasets maestros exportados a Parquet

---
### Fuentes de datos (`datos_final/`)
| # | Archivo | Formato | Contenido |
|---|---|---|---|
| 1 | `partidos_historicos_ligamx.csv` | CSV | 4 080 partidos Liga MX 2010–2024 |
| 2 | `perfiles_jugadores_historico.json` | JSON anidado | 1 150 jugadores con datos de 3 sub-fuentes |
| 3 | `scouting_historico_ligamx.txt` | TXT ancho fijo | Reporte de scouting con métricas por temporada |
| 4 | `valor_mercado_historico.xlsx` | XLSX | Valores de mercado e inconsistencias etiquetadas |

### Problemáticas que esta etapa atiende
| Prob. | Descripción | Dimensión DAMA |
|---|---|---|
| #1 | Nombres distintos del mismo jugador entre fuentes | Consistencia |
| #3 | Jugadores con doble nacionalidad registrados dos veces | Unicidad |
| #4 | Equipos renombrados entre temporadas | Integridad referencial |

In [ ]:
import os
import io
import json
import re
import unicodedata
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from rapidfuzz import fuzz

# ── Rutas ────────────────────────────────────────────────────
BASE   = os.path.abspath(os.path.join(os.getcwd(), '..'))
DATA   = os.path.join(BASE, 'datos_final')
MASTER = os.path.join(BASE, 'datos_master')
os.makedirs(MASTER, exist_ok=True)

# Paleta Cruz Azul
AZUL, ROJO, GRIS = '#003DA5', '#C8102E', '#888888'
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.titlesize'] = 13
sns.set_style('whitegrid')

print('BASE   :', BASE)
print('DATA   :', DATA)
print('MASTER :', MASTER)

---
## 1. Carga y Estandarización de Fuentes

Cada fuente viene en un formato diferente. Los convertimos a DataFrames tabulares con columnas homogéneas antes de cualquier procesamiento posterior.

In [ ]:
# ── Fuente 1: Partidos históricos (CSV) ──────────────────────
df_partidos = pd.read_csv(os.path.join(DATA, 'partidos_historicos_ligamx.csv'))
df_partidos['date'] = pd.to_datetime(df_partidos['date'], errors='coerce')

print(f'Fuente 1 — Partidos CSV : {df_partidos.shape[0]:,} filas × {df_partidos.shape[1]} columnas')
display(df_partidos.head(3))

In [ ]:
# ── Fuente 2: Perfiles de jugadores (JSON anidado) ───────────
with open(os.path.join(DATA, 'perfiles_jugadores_historico.json'), encoding='utf-8') as f:
    raw_json = json.load(f)

print('Metadata del dataset:')
for k, v in raw_json['metadata'].items():
    if not isinstance(v, (dict, list)):
        print(f'  {k}: {v}')

# Aplanar la estructura anidada: jugadores[*] -> fila del DataFrame
rows = []
for j in raw_json['jugadores']:
    consolidado = j['datos_consolidados']
    rendimiento = consolidado.pop('rendimiento', {})
    row = {
        'player_id'          : j['player_id'],
        # Nombres por fuente (inconsistencia #1)
        'nombre_football_csv': j['fuentes']['football_csv']['nombre'],
        'nombre_transfermarkt': j['fuentes']['transfermarkt']['nombre'],
        'nombre_fifa'        : j['fuentes']['fifa']['nombre'],
        # Atributos FIFA
        'fifa_overall'       : j['fuentes']['fifa'].get('overall'),
        'fifa_pace'          : j['fuentes']['fifa'].get('pace'),
        'fifa_shooting'      : j['fuentes']['fifa'].get('shooting'),
        'fifa_passing'       : j['fuentes']['fifa'].get('passing'),
        'fifa_dribbling'     : j['fuentes']['fifa'].get('dribbling'),
        'fifa_defending'     : j['fuentes']['fifa'].get('defending'),
        'fifa_physical'      : j['fuentes']['fifa'].get('physical'),
        # Valores Transfermarkt
        'valor_mercado_eur'  : j['fuentes']['transfermarkt'].get('valor_mercado_eur'),
        'transfer_paid_eur'  : j['fuentes']['transfermarkt'].get('transfer_paid_eur'),
        'tm_nota'            : j['fuentes']['transfermarkt'].get('nota', ''),
        # Datos consolidados
        **consolidado,
        # Rendimiento
        **rendimiento,
        # Inconsistencias etiquetadas
        'inconsistencias'    : '|'.join(j.get('inconsistencias_detectadas', []))
    }
    rows.append(row)

df_jugadores = pd.DataFrame(rows)
print(f'\nFuente 2 — Jugadores JSON: {df_jugadores.shape[0]:,} filas × {df_jugadores.shape[1]} columnas')
display(df_jugadores.head(3))

In [ ]:
# ── Fuente 3: Scouting (TXT ancho fijo) ──────────────────────
# El archivo tiene 6 líneas de cabecera, luego encabezado de columnas,
# separador y datos. Se parsea con read_fwf.
lines_raw = open(os.path.join(DATA, 'scouting_historico_ligamx.txt'),
                 encoding='utf-8').readlines()

header_line = lines_raw[7]   # línea con nombres de columnas
data_lines  = [l for l in lines_raw[9:]
               if l.strip() and l.strip()[0] == 'P' and l.strip()[1:5].isdigit()]

df_scouting = pd.read_fwf(
    io.StringIO(header_line + ''.join(data_lines)),
    header=0
)
# Renombrar columnas al esquema interno
col_map = {
    'ID'     : 'player_id',
    'NOMBRE_TM'  : 'sc_nombre_tm',
    'NOMBRE_FIFA': 'sc_nombre_fifa',
    'NOMBRE_CSV' : 'sc_nombre_csv',
    'POS'    : 'sc_posicion',
    'NAC          EXT': 'sc_nac_ext',   # columna fusionada por fwf
    'EDAD'   : 'sc_edad',
    'TEMP           EQUIPO': 'sc_temp_equipo',
    'OVR'    : 'sc_overall',
    'MVAL_EUR'   : 'sc_mval_eur',
    'TPAG_EUR'   : 'sc_tpag_eur',
    'GOLES'  : 'sc_goles',
    'ASIS'   : 'sc_asistencias',
    'MINS'   : 'sc_minutos',
    'G+A/90' : 'sc_ga_90',
}
df_scouting.rename(columns={k: v for k, v in col_map.items()
                             if k in df_scouting.columns}, inplace=True)
# Limpiar columnas residuales del parseo de ancho fijo
df_scouting.drop(columns=[c for c in df_scouting.columns
                           if c.startswith('Unnamed')], inplace=True)
# Limpiar valores numéricos con comas (ej. '750,000' -> 750000)
for col in ['sc_mval_eur', 'sc_tpag_eur']:
    if col in df_scouting.columns:
        df_scouting[col] = (df_scouting[col].astype(str)
                            .str.replace(',', '', regex=False)
                            .pipe(pd.to_numeric, errors='coerce'))

print(f'Fuente 3 — Scouting TXT: {df_scouting.shape[0]:,} filas × {df_scouting.shape[1]} columnas')
display(df_scouting.head(3))

In [ ]:
# ── Fuente 4: Valor de mercado (XLSX) ────────────────────────
df_xlsx = pd.read_excel(os.path.join(DATA, 'valor_mercado_historico.xlsx'))

print(f'Fuente 4 — Valor mercado XLSX: {df_xlsx.shape[0]:,} filas × {df_xlsx.shape[1]} columnas')
display(df_xlsx.head(3))
print()
print('Columnas disponibles:', df_xlsx.columns.tolist())

---
## 2. Modelo Canónico

Definimos los esquemas **canónicos** que usarán todos los notebooks posteriores.  
El modelo canónico es la fuente de verdad: cada campo tiene **una** definición, **un** tipo y **un** nombre.

### 2.1 Modelo Canónico de Jugador
| Campo canónico | Fuente de verdad | Descripción |
|---|---|---|
| `player_id` | JSON / XLSX | Clave primaria (P0001…P1150) |
| `nombre_canonico` | Transfermarkt (prioridad) | Nombre completo normalizado |
| `posicion` | JSON consolidado | GK, CB, RB, CM, CAM, ST… |
| `nacionalidad` | JSON consolidado | País principal |
| `es_extranjero` | JSON consolidado | Bool |
| `doble_nacionalidad` | JSON consolidado | Bool |
| `edad` | JSON consolidado | Años enteros |
| `temporada` | JSON consolidado | Ej. "Apertura 2013" |
| `equipo` | JSON consolidado | Nombre normalizado |
| `fifa_overall` | FIFA (escala 0-100) | Rating FIFA |
| `valor_mercado_eur` | Transfermarkt | Valor en euros |
| `goles` | football.csv / JSON | Entero |
| `asistencias` | football.csv / JSON | Entero |
| `minutos` | football.csv / JSON | Entero |
| `g_a_por_90` | calculado | (goles+asist)/90 min |

### 2.2 Modelo Canónico de Partido
| Campo canónico | Descripción |
|---|---|
| `match_id` | Clave primaria |
| `date` | Fecha del partido |
| `season` | Ej. "Apertura 2010" |
| `season_year` | Año numérico |
| `half` | apertura / clausura |
| `equipo_local_norm` | Nombre canónico local |
| `equipo_visitante_norm` | Nombre canónico visitante |
| `goles_local` | Int |
| `goles_visitante` | Int |
| `resultado_local` | W / D / L |
| `involucra_cruz_azul` | Bool |

In [ ]:
# ── Funciones de normalización canónica ──────────────────────

def normalizar_texto(texto):
    """Minúsculas, sin acentos, solo letras y espacios, espacios colapsados."""
    if pd.isna(texto):
        return np.nan
    s = str(texto).strip().lower()
    s = unicodedata.normalize('NFD', s)
    s = ''.join(c for c in s if unicodedata.category(c) != 'Mn')
    s = re.sub(r'[^a-z0-9 ]', ' ', s)
    return re.sub(r'\s+', ' ', s).strip()


# Diccionario de alias de equipos → nombre canónico
ALIAS_EQUIPOS = {
    'cruz azul'          : 'CRUZ AZUL',
    'la maquina'         : 'CRUZ AZUL',
    'america'            : 'CLUB AMERICA',
    'aguilas'            : 'CLUB AMERICA',
    'club america'       : 'CLUB AMERICA',
    'guadalajara'        : 'CHIVAS GUADALAJARA',
    'chivas'             : 'CHIVAS GUADALAJARA',
    'deportivo guadalajara': 'CHIVAS GUADALAJARA',
    'tigres'             : 'TIGRES UANL',
    'uanl tigres'        : 'TIGRES UANL',
    'monterrey'          : 'MONTERREY',
    'rayados'            : 'MONTERREY',
    'pumas'              : 'PUMAS UNAM',
    'unam'               : 'PUMAS UNAM',
    'atlas'              : 'ATLAS',
    'atlas guadalajara'  : 'ATLAS',
    'necaxa'             : 'NECAXA',
    'impulsora necaxa'   : 'NECAXA',
    'santos'             : 'SANTOS LAGUNA',
    'santos laguna'      : 'SANTOS LAGUNA',
    'toluca'             : 'TOLUCA',
    'pachuca'            : 'PACHUCA',
    'leon'               : 'LEON',
    'tijuana'            : 'TIJUANA',
    'club tijuana'       : 'TIJUANA',
    'xolos'              : 'TIJUANA',
    'mazatlan'           : 'MAZATLAN FC',
    'mazatlan fc'        : 'MAZATLAN FC',
    # Renombre: Monarcas Morelia → Mazatlán FC (2020) — Problemática #4
    'monarcas morelia'   : 'MAZATLAN FC',
    'morelia'            : 'MAZATLAN FC',
    # Renombre: Dorados Sinaloa → Mazatlán FC (2020)
    'dorados sinaloa'    : 'MAZATLAN FC',
    'dorados'            : 'MAZATLAN FC',
    'puebla'             : 'PUEBLA',
    'puebla fc'          : 'PUEBLA',
    'fc juarez'          : 'FC JUAREZ',
    'juarez'             : 'FC JUAREZ',
    'queretaro'          : 'QUERETARO',
    'gallos blancos'     : 'QUERETARO',
    'atletico san luis'  : 'ATLETICO SAN LUIS',
    'san luis'           : 'ATLETICO SAN LUIS',
}

STOPWORDS = [r'\bf\.?c\.?\b', r'\bc\.?f\.?\b', r'\ba\.?c\.?\b',
             r'\bde\b', r'\blos\b', r'\blas\b', r'\bel\b', r'\bla\b']

def normalizar_equipo(nombre):
    if pd.isna(nombre): return np.nan
    s = normalizar_texto(nombre)
    for pat in STOPWORDS:
        s = re.sub(pat, '', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return ALIAS_EQUIPOS.get(s, s.upper())


def normalizar_nombre(texto):
    r = normalizar_texto(texto)
    return r.upper() if isinstance(r, str) else np.nan


print('Funciones de normalización cargadas.')

---
## 3. Record Linkage — Reconciliación de Nombres de Jugadores

### Problemática #1
El mismo jugador aparece con **tres variantes de nombre** según la fuente:
- `NOMBRE_TM` (Transfermarkt): nombre completo → *Jonathan García*
- `NOMBRE_FIFA` (FIFA 23): abreviado → *J. García*
- `NOMBRE_CSV` (football.csv): invertido → *García J.*

Como el JSON ya provee el `player_id` como clave, usamos **similitud de cadenas** para:  
a) Cuantificar el grado de inconsistencia entre fuentes  
b) Elegir el **nombre canónico** (Transfermarkt como fuente de verdad)  
c) Detectar casos donde el player_id podría estar mal asignado

In [ ]:
# Calcular similitud Jaro-Winkler entre pares de nombres por jugador
def sim_jw(a, b):
    """Jaro-Winkler normalizado en [0, 1]. Devuelve NaN si alguno es vacío."""
    if pd.isna(a) or pd.isna(b):
        return np.nan
    return fuzz.WRatio(str(a), str(b)) / 100

df_jugadores['sim_tm_fifa'] = df_jugadores.apply(
    lambda r: sim_jw(r['nombre_transfermarkt'], r['nombre_fifa']), axis=1)
df_jugadores['sim_tm_csv']  = df_jugadores.apply(
    lambda r: sim_jw(r['nombre_transfermarkt'], r['nombre_football_csv']), axis=1)
df_jugadores['sim_fifa_csv'] = df_jugadores.apply(
    lambda r: sim_jw(r['nombre_fifa'], r['nombre_football_csv']), axis=1)

print('Estadísticas de similitud entre nombres por fuente:')
display(df_jugadores[['sim_tm_fifa', 'sim_tm_csv', 'sim_fifa_csv']].describe().round(3))

# Casos con baja similitud (potenciales errores de asignación de player_id)
umbral = 0.6
casos_bajos = df_jugadores[
    (df_jugadores['sim_tm_fifa'] < umbral) |
    (df_jugadores['sim_tm_csv']  < umbral)
][['player_id', 'nombre_transfermarkt', 'nombre_fifa',
   'nombre_football_csv', 'sim_tm_fifa', 'sim_tm_csv']]

print(f'\nJugadores con similitud < {umbral} en al menos un par: {len(casos_bajos)}')
display(casos_bajos.head(10))

In [ ]:
# Visualización: distribución de similitud entre fuentes
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
pares = [
    ('sim_tm_fifa',  'TM vs FIFA',          AZUL),
    ('sim_tm_csv',   'TM vs football.csv',  ROJO),
    ('sim_fifa_csv', 'FIFA vs football.csv', GRIS),
]
for ax, (col, titulo, color) in zip(axes, pares):
    df_jugadores[col].dropna().hist(bins=30, ax=ax, color=color, edgecolor='white')
    media = df_jugadores[col].mean()
    ax.axvline(media, color='black', linestyle='--', linewidth=1.2,
               label=f'Media: {media:.2f}')
    ax.set_title(f'Similitud {titulo}', fontweight='bold')
    ax.set_xlabel('Score Jaro-Winkler')
    ax.set_ylabel('Jugadores')
    ax.legend(fontsize=9)

plt.suptitle('Problemática #1 — Inconsistencia de nombres entre fuentes\n'
             '(score < 0.6 indica discrepancia grave)',
             fontweight='bold', fontsize=12)
plt.tight_layout()
plt.show()

# Cuantificación para el reporte
for col, label in [('sim_tm_fifa','TM vs FIFA'), ('sim_tm_csv','TM vs CSV')]:
    pct = (df_jugadores[col] < 0.6).mean() * 100
    print(f'  {label}: {pct:.1f}% de jugadores con similitud < 0.6')

In [ ]:
# Definir nombre canónico: Transfermarkt es la fuente de verdad
# Si está vacío, se usa FIFA; si también está vacío, football.csv
df_jugadores['nombre_canonico'] = (
    df_jugadores['nombre_transfermarkt']
    .fillna(df_jugadores['nombre_fifa'])
    .fillna(df_jugadores['nombre_football_csv'])
)
df_jugadores['nombre_key'] = df_jugadores['nombre_canonico'].apply(normalizar_nombre)

resolucion = pd.DataFrame({
    'Fuente de verdad': ['Transfermarkt', 'FIFA (fallback)', 'football.csv (fallback)', 'Sin nombre'],
    'Registros': [
        df_jugadores['nombre_transfermarkt'].notna().sum(),
        (df_jugadores['nombre_transfermarkt'].isna() & df_jugadores['nombre_fifa'].notna()).sum(),
        (df_jugadores['nombre_transfermarkt'].isna() & df_jugadores['nombre_fifa'].isna() &
         df_jugadores['nombre_football_csv'].notna()).sum(),
        df_jugadores['nombre_canonico'].isna().sum(),
    ]
})
print('Resolución de nombre canónico:')
display(resolucion)
print(f'\nEjemplos de nombre canónico asignado:')
display(df_jugadores[['player_id', 'nombre_transfermarkt', 'nombre_fifa',
                       'nombre_football_csv', 'nombre_canonico']].head(8))

---
## 4. Deduplicación

### Problemática #3 — Doble nacionalidad
Jugadores con doble ciudadanía (ej. MEX/ESP) pueden aparecer registrados **dos veces** en análisis de cupo de extranjeros: una como mexicano y otra como extranjero. Esto infla el catálogo y distorsiona los límites de plantilla.

### Problemática #4 — Renombre de equipos
Monarcas Morelia se convirtió en Mazatlán FC en 2020, y Dorados Sinaloa también fue absorbido. Sin normalización, se generan series históricas fragmentadas.

In [ ]:
# ── Deduplicación por doble nacionalidad (Problemática #3) ───
doble_nac = df_jugadores[df_jugadores['doble_nacionalidad'] == True].copy()
print(f'Jugadores con doble nacionalidad: {len(doble_nac)}')

# Detectar si el mismo jugador (mismo nombre_key) aparece dos veces
# con distinto player_id por error de catalogación
dup_nombres = df_jugadores[df_jugadores.duplicated(subset='nombre_key', keep=False)]
print(f'Registros con nombre_key duplicado entre distintos player_id: {len(dup_nombres)}')
if len(dup_nombres) > 0:
    display(dup_nombres[['player_id', 'nombre_canonico', 'nacionalidad',
                          'segunda_nacionalidad', 'doble_nacionalidad',
                          'temporada', 'equipo']].head(10))

# Por diseño del dataset, el player_id es único; los duplicados de cupo
# se gestionan con la columna doble_nac del XLSX
doble_nac_xlsx = df_xlsx[df_xlsx['doble_nac'] == True]
print(f'\nRegistros XLSX con doble_nac=True: {len(doble_nac_xlsx)}')
display(doble_nac_xlsx[['player_id', 'nombre_tm', 'nombre_fifa',
                          'nacionalidad', 'segunda_nac',
                          'es_extranjero', 'doble_nac']].head(8))

In [ ]:
# ── Normalización de equipos — Problemática #4 ───────────────
df_partidos['equipo_local_norm']     = df_partidos['home_team'].apply(normalizar_equipo)
df_partidos['equipo_visitante_norm'] = df_partidos['away_team'].apply(normalizar_equipo)

df_jugadores['equipo_norm'] = df_jugadores['equipo'].apply(
    lambda x: normalizar_equipo(str(x).split()[-1] + ' ' + ' '.join(str(x).split()[:-1])
              if pd.notna(x) else x)
)
# Corrección directa usando normalizar_equipo sobre el campo completo
df_jugadores['equipo_norm'] = df_jugadores['equipo'].apply(normalizar_equipo)

# Mostrar el impacto de la normalización en partidos
antes = df_partidos['home_team'].nunique()
despues = df_partidos['equipo_local_norm'].nunique()
print(f'Equipos únicos ANTES de normalizar : {antes}')
print(f'Equipos únicos DESPUÉS de normalizar: {despues}')
print(f'Reducción: {antes - despues} nombres consolidados')

# Casos donde el nombre cambió (renombres detectados)
renombres = df_partidos[df_partidos['home_team'].apply(normalizar_texto) !=
                        df_partidos['equipo_local_norm'].apply(normalizar_texto)]
print(f'\nPartidos con equipo local renombrado: {len(renombres)}')
display(renombres[['date', 'home_team', 'equipo_local_norm']]
        .drop_duplicates(subset='home_team').head(10))

In [ ]:
# Visualizar equipos con renombre histórico (Problemática #4)
renombre_equipos = {
    'Monarcas Morelia'  : 'MAZATLAN FC (desde 2020)',
    'Dorados Sinaloa'   : 'MAZATLAN FC (desde 2020)',
}
fig, ax = plt.subplots(figsize=(12, 4))
for equipo_viejo, equipo_nuevo in renombre_equipos.items():
    mask = df_partidos['home_team'].str.contains(equipo_viejo.split()[0], case=False, na=False)
    conteo = df_partidos[mask].groupby('season_year').size()
    if not conteo.empty:
        ax.bar(conteo.index.astype(str), conteo.values,
               label=f'{equipo_viejo} → {equipo_nuevo}', alpha=0.8)

ax.axvline('2020', color='black', linestyle='--', linewidth=1.5, label='Año del renombre (2020)')
ax.set_title('Problemática #4 — Partidos registrados bajo nombres históricos',
             fontweight='bold')
ax.set_xlabel('Temporada')
ax.set_ylabel('Partidos como local')
ax.legend()
plt.tight_layout()
plt.show()

---
## 5. Fusión — Construcción de Datasets Maestros

Combinamos la información de todas las fuentes en **dos datasets maestros**:
- `master_jugadores.parquet` — Un registro por (player_id, temporada)
- `master_partidos.parquet` — Un registro por partido

La clave de unión es `player_id` para jugadores y `match_id` para partidos.

In [ ]:
# ── Master Jugadores ─────────────────────────────────────────
# Base: JSON (tiene datos de las 3 sub-fuentes + consolidado)
# Enriquecimiento: XLSX (columnas adicionales de rendimiento)

# Columnas del XLSX que aportan información extra no presente en JSON
cols_xlsx_extra = ['player_id', 'p_casa', 'goles_casa',
                   'p_fuera', 'goles_fuera', 'tarj_am', 'tarj_roj']
cols_xlsx_extra = [c for c in cols_xlsx_extra if c in df_xlsx.columns]

# Merge JSON + XLSX (left join: preservar todos los registros del JSON)
master_jug = df_jugadores.merge(
    df_xlsx[cols_xlsx_extra],
    on='player_id',
    how='left',
    suffixes=('', '_xlsx')
)

# Columnas canónicas finales (selección y orden)
cols_master_jug = [
    'player_id', 'nombre_canonico', 'nombre_key',
    'nombre_transfermarkt', 'nombre_fifa', 'nombre_football_csv',
    'sim_tm_fifa', 'sim_tm_csv',
    'posicion', 'nacionalidad', 'segunda_nacionalidad',
    'es_extranjero', 'doble_nacionalidad',
    'edad', 'temporada', 'season_year', 'equipo', 'equipo_norm',
    'fifa_overall', 'fifa_pace', 'fifa_shooting', 'fifa_passing',
    'fifa_dribbling', 'fifa_defending', 'fifa_physical',
    'valor_mercado_eur', 'transfer_paid_eur',
    'goles', 'asistencias', 'minutos', 'partidos', 'g_a_por_90',
    'p_casa', 'goles_casa', 'p_fuera', 'goles_fuera',
    'tarj_am', 'tarj_roj',
    'inconsistencias',
]
cols_master_jug = [c for c in cols_master_jug if c in master_jug.columns]
master_jug = master_jug[cols_master_jug].copy()

print(f'Master Jugadores: {master_jug.shape[0]:,} filas × {master_jug.shape[1]} columnas')
display(master_jug.head(3))

In [ ]:
# ── Master Partidos ──────────────────────────────────────────
# Aplicar modelo canónico al CSV de partidos
master_par = df_partidos.rename(columns={
    'home_team' : 'equipo_local',
    'away_team' : 'equipo_visitante',
    'home_goals': 'goles_local',
    'away_goals': 'goles_visitante',
    'home_goals_ht': 'goles_local_ht',
    'away_goals_ht': 'goles_visitante_ht',
    'result_home': 'resultado_local',
    'involves_cruz_azul': 'involucra_cruz_azul',
    'inconsistency_note': 'nota_inconsistencia',
}).copy()

# Columnas canónicas finales
cols_master_par = [
    'match_id', 'date', 'season', 'season_year', 'half', 'round',
    'home_team_id', 'equipo_local', 'equipo_local_norm',
    'away_team_id', 'equipo_visitante', 'equipo_visitante_norm',
    'goles_local', 'goles_visitante',
    'goles_local_ht', 'goles_visitante_ht',
    'venue', 'resultado_local', 'involucra_cruz_azul', 'nota_inconsistencia'
]
cols_master_par = [c for c in cols_master_par if c in master_par.columns]
master_par = master_par[cols_master_par].copy()

print(f'Master Partidos: {master_par.shape[0]:,} filas × {master_par.shape[1]} columnas')
display(master_par.head(3))

In [ ]:
# ── Exportar a Parquet ───────────────────────────────────────
ruta_jug = os.path.join(MASTER, 'master_jugadores.parquet')
ruta_par = os.path.join(MASTER, 'master_partidos.parquet')

master_jug.to_parquet(ruta_jug, index=False)
master_par.to_parquet(ruta_par, index=False)

print(f'master_jugadores.parquet  → {ruta_jug}')
print(f'master_partidos.parquet   → {ruta_par}')
print(f'\nJugadores : {len(master_jug):,} registros | {master_jug.memory_usage(deep=True).sum()/1024:.1f} KB')
print(f'Partidos  : {len(master_par):,} registros | {master_par.memory_usage(deep=True).sum()/1024:.1f} KB')

---
## 6. Diagnóstico de Cobertura

Verificamos que la fusión no perdió registros y cuantificamos los casos problemáticos.

In [ ]:
# Cobertura del merge JSON + XLSX
xlsx_ids = set(df_xlsx['player_id'])
json_ids = set(df_jugadores['player_id'])

print('Diagnóstico de cobertura — Jugadores')
print(f'  player_id en JSON              : {len(json_ids):,}')
print(f'  player_id en XLSX              : {len(xlsx_ids):,}')
print(f'  IDs en JSON pero no en XLSX    : {len(json_ids - xlsx_ids)}')
print(f'  IDs en XLSX pero no en JSON    : {len(xlsx_ids - json_ids)}')
print(f'  IDs presentes en ambos         : {len(json_ids & xlsx_ids):,}')
print(f'\nDiagnóstico de cobertura — Partidos')
print(f'  Partidos totales en CSV        : {len(df_partidos):,}')
print(f'  Temporadas cubiertas           : {df_partidos["season_year"].nunique()}')
print(f'  Equipos únicos (normalizado)   : {master_par["equipo_local_norm"].nunique()}')
print(f'  Partidos con nota inconsistencia: {master_par["nota_inconsistencia"].notna().sum()}')

In [ ]:
# Resumen visual de cobertura por temporada
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Jugadores por temporada
conteo_temp_jug = master_jug['season_year'].value_counts().sort_index()
axes[0].bar(conteo_temp_jug.index.astype(str), conteo_temp_jug.values, color=AZUL)
axes[0].set_title('Jugadores en master por temporada', fontweight='bold')
axes[0].set_xlabel('Año')
axes[0].set_ylabel('Registros')
axes[0].tick_params(axis='x', rotation=45)

# Partidos por temporada
conteo_temp_par = master_par['season_year'].value_counts().sort_index()
axes[1].bar(conteo_temp_par.index.astype(str), conteo_temp_par.values, color=ROJO)
axes[1].set_title('Partidos en master por temporada', fontweight='bold')
axes[1].set_xlabel('Año')
axes[1].set_ylabel('Partidos')
axes[1].tick_params(axis='x', rotation=45)

plt.suptitle('Cobertura temporal de los datasets maestros', fontweight='bold')
plt.tight_layout()
plt.show()

---
## 7. Conclusiones de la Etapa de Integración

| Problemática | Acción realizada | Resultado |
|---|---|---|
| #1 — Nombres inconsistentes | Record linkage con Jaro-Winkler; nombre canónico = Transfermarkt | `nombre_canonico` unificado para todos los registros |
| #3 — Doble nacionalidad | Detección con flag `doble_nacionalidad`; no se eliminan registros (decisión de negocio) | Columna `doble_nac` disponible para filtrar en análisis |
| #4 — Renombres de equipos | Diccionario de alias + normalización → Monarcas/Dorados → MAZATLAN FC | Reducción de nombres únicos; series históricas continuas |

**Outputs generados:**
- `datos_master/master_jugadores.parquet` — Dataset maestro de jugadores
- `datos_master/master_partidos.parquet` — Dataset maestro de partidos

**Siguiente etapa:** `perfilado.ipynb` — Perfilado previo a la limpieza sobre los datos maestros.